In [ ]:
# !pip install geopandas pystac-client planetary-computer rioxarray matplotlib pandas tqdm

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pystac_client
import planetary_computer
import rioxarray
from shapely.geometry import mapping
from tqdm import tqdm

# Initialize the Planetary Computer STAC Client
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

In [ ]:
# 1. Load the Geopackage
gpkg_path = "path_to_your_refineries.gpkg" 
refineries = gpd.read_file(gpkg_path)

# Ensure data is in a geographic coordinate reference system (WGS84) for the STAC API
if refineries.crs != "EPSG:4326":
    refineries = refineries.to_crs("EPSG:4326")

# Extract the overall bounding box (xmin, ymin, xmax, ymax)
bbox = list(refineries.total_bounds)
print(f"Loaded {len(refineries)} refinery boundaries.")
print(f"Search bounding box: {bbox}")

In [ ]:
# Define temporal window
start_date = "2010-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")
time_range = f"{start_date}/{end_date}"

print(f"Searching for active fire anomalies between {time_range}...")

search = catalog.search(
    collections=["modis-14A1-061"],
    bbox=bbox,
    datetime=time_range
)

items = search.item_collection()
print(f"Found {len(items)} STAC items matching criteria.")

In [ ]:
fire_records = []

# Loop over discovered STAC items
for item in tqdm(items, desc="Processing Fire Data"):
    # Target observation dates
    item_date = pd.to_datetime(item.properties["datetime"]).date()
    
    # Get signed asset URL for the Fire Mask
    try:
        asset_url = item.assets["FireMask"].href
        
        # Load cloud geotiff without downloading the full tile
        with rioxarray.open_rasterio(asset_url) as rxs:
            # Clip the raster strictly to your refinery geometry boundaries
            clipped = rxs.rio.clip(refineries.geometry.apply(mapping), refineries.crs, drop=True)
            
            # Values 7, 8, and 9 represent confirmed active fire pixels
            # (Class 7 = Low, 8 = Nominal, 9 = High confidence fire)
            fire_pixels = np.isin(clipped.values, [7, 8, 9])
            fire_count = int(np.sum(fire_pixels))
            
            if fire_count > 0:
                fire_records.append({
                    "date": item_date,
                    "fire_pixel_count": fire_count
                })
    except Exception as e:
        # Gracefully handle missing assets or edge clipping errors
        continue

# Convert results into a structured Pandas DataFrame
if fire_records:
    df_fires = pd.DataFrame(fire_records)
    df_fires['date'] = pd.to_datetime(df_fires['date'])
    df_fires = df_fires.sort_values('date').set_index('date')
    print(f"\nExtraction complete. Detected thermal anomalies on {len(df_fires)} unique days.")
else:
    df_fires = pd.DataFrame(columns=["fire_pixel_count"])
    print("\nNo active thermal anomalies detected inside your boundaries during this timeframe.")

In [ ]:
if not df_fires.empty:
    # Resample to make sure every month/day is accounted for in the timeseries
    df_daily = df_fires.resample('D').sum().fillna(0)
    df_monthly = df_fires.resample('ME').sum().fillna(0) # Monthly aggregation for smoother trends

    plt.figure(figsize=(14, 6))
    
    # Plot raw daily counts as translucent stems/bars
    plt.bar(df_daily.index, df_daily['fire_pixel_count'], color='orangered', alpha=0.5, label='Daily Fire Pixels')
    
    # Plot smooth monthly trendline
    plt.plot(df_monthly.index, df_monthly['fire_pixel_count'], color='darkred', linewidth=2, label='Monthly Aggregate Trend')
    
    plt.title('Thermal Anomaly History & Fire Detection inside Refinery Boundaries (2010 - Present)', fontsize=14, fontweight='bold')
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Detected Active Fire Pixel Volume', fontsize=12)
    plt.xlim(pd.to_datetime("2010-01-01"), pd.to_datetime(datetime.now()))
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(loc='upper left', fontsize=11)
    
    plt.tight_layout()
    plt.show()
else:
    print("Plotting skipped because no data entries were found within specified geometries.")